In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

In [2]:
spark = SparkSession.builder.appName("PySpark_Pipeline_Mastery").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 13:33:31 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/24 13:33:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 13:33:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/24 13:33:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/24 13:33:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/24 13:33:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
data = [
    # age, city, clicks_ads (Target)
    (25.0, "New York", 1.0),
    (None, "Chicago",  0.0), # Missing age
    (34.0, "New York", 1.0),
    (45.0, "Boston",   1.0),
    (None, "Chicago",  0.0), # Missing age
    (22.0, "Boston",   0.0),
    (50.0, "New York", 1.0),
    (38.0, "Boston",   0.0)
]

df = spark.createDataFrame(data, ["age", "city", "clicks_ads"])
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

print("Raw Training Data:")
train_df.show()

Raw Training Data:


+----+--------+----------+
| age|    city|clicks_ads|
+----+--------+----------+
|NULL| Chicago|       0.0|
|34.0|New York|       1.0|
|45.0|  Boston|       1.0|
|NULL| Chicago|       0.0|
|50.0|New York|       1.0|
+----+--------+----------+



In [4]:
# Stage A: Handle Nulls in Age (Estimator)
imputer = Imputer(
    inputCols=["age"], 
    outputCols=["age_imputed"]
).setStrategy("median")

# Stage B: Convert City strings to numbers (Estimator)
indexer = StringIndexer(
    inputCol="city", 
    outputCol="city_index", 
    handleInvalid="keep"
)

# Stage C: Assemble features into a vector (Transformer)
# Notice we use the OUTPUT columns from previous stages!
assembler = VectorAssembler(
    inputCols=["age_imputed", "city_index"], 
    outputCol="features"
)

# Stage D: The Machine Learning Model (Estimator)
rf = RandomForestClassifier(
    featuresCol="features", 
    labelCol="clicks_ads", 
    numTrees=10, 
    maxBins=32 # High enough to cover all unique cities
)

In [5]:
pipeline = Pipeline(stages=[imputer, indexer, assembler, rf])
pipeline_model = pipeline.fit(train_df)

26/06/24 13:33:41 WARN DecisionTreeMetadata: DecisionTree reducing maxBins from 32 to 5 (= number of training instances)


In [6]:
predictions = pipeline_model.transform(test_df)

print("Final Predictions:")
predictions.select("age", "city", "features", "clicks_ads", "prediction").show(truncate=False)

Final Predictions:
+----+--------+----------+----------+----------+
|age |city    |features  |clicks_ads|prediction|
+----+--------+----------+----------+----------+
|25.0|New York|[25.0,1.0]|1.0       |1.0       |
|22.0|Boston  |[22.0,2.0]|0.0       |1.0       |
|38.0|Boston  |[38.0,2.0]|0.0       |1.0       |
+----+--------+----------+----------+----------+



In [7]:
spark.stop()